## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-19-14-51-00 +0000,wsj,KKR Invests in India Renewables Business Run b...,https://www.wsj.com/pro/private-equity/kkr-inv...
1,2026-11-19-14-50-16 +0000,nyt,"Live Updates: Trump, Xi and Putin Seek to Proj...",https://www.nytimes.com/live/2026/11/18/world/...
2,2026-11-19-14-50-12 +0000,nyt,Inca Children Were Given Cocaine and Ayahuasca...,https://www.nytimes.com/2026/11/18/science/inc...
3,2026-11-19-14-48-31 +0000,nyt,How Anthropic’s Dario Amodei thinks about A.I.,https://www.nytimes.com/live/2026/11/19/techno...
4,2026-11-19-14-43-56 +0000,nyt,When will Anthropic shares start trading?,https://www.nytimes.com/live/2026/11/19/techno...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2520/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
11,trump,55
29,anthropic,27
13,putin,25
375,new,21
12,xi,20


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
394,2026-11-18-18-41-56 +0000,chitri,President Trump meets Xi Jinping and Vladimir ...,https://www.chicagotribune.com/2026/11/18/trum...,154
201,2026-11-19-03-53-50 +0000,chitri,President Trump meets Xi and Putin in China wi...,https://www.chicagotribune.com/2026/11/18/trum...,152
136,2026-11-19-10-24-31 +0000,latimes,"'Making history': Trump, Putin and Xi meet in ...",https://www.latimes.com/politics/story/2026-11...,148
1,2026-11-19-14-50-16 +0000,nyt,"Live Updates: Trump, Xi and Putin Seek to Proj...",https://www.nytimes.com/live/2026/11/18/world/...,147
215,2026-11-19-02-44-01 +0000,nypost,Kim Jong Un’s nuclear program draws ‘serious c...,https://nypost.com/2026/11/18/world-news/kim-j...,143


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
394,154,2026-11-18-18-41-56 +0000,chitri,President Trump meets Xi Jinping and Vladimir ...,https://www.chicagotribune.com/2026/11/18/trum...
217,91,2026-11-19-02-00-00 +0000,wsj,Jensen Huang’s ouster from Nvidia is giving ne...,https://www.wsj.com/politics/national-security...
307,84,2026-11-18-22-13-12 +0000,cbc,She says Russell Brand sexually assaulted her ...,https://www.cbc.ca/radio/asithappens/mia-latte...
344,74,2026-11-18-20-56-05 +0000,wapo,Democrat Hakeem Jeffries nominated to be first...,https://www.washingtonpost.com/politics/2026/1...
235,72,2026-11-19-01-11-00 +0000,wsj,Elon Musk said he felt “pretty foolish” about ...,https://www.wsj.com/politics/policy/elon-musk-...
6,61,2026-11-19-14-42-03 +0000,nyt,Anthropic’s Public Debut Sets Up ‘Experiment’ ...,https://www.nytimes.com/2026/11/18/business/an...
150,52,2026-11-19-09-50-40 +0000,cbc,South Korea's top court overturns charges agai...,https://www.cbc.ca/news/world/south-korea-over...
280,47,2026-11-18-23-09-09 +0000,nypost,Shocking video shows Putin’s prized Crimea bri...,https://nypost.com/2026/11/18/world-news/commu...
120,46,2026-11-19-11-00-00 +0000,latimes,"High tides, big storms: Next week will be key ...",https://www.latimes.com/california/story/2026-...
300,42,2026-11-18-22-28-09 +0000,chitri,Cook County committee extends troubled propert...,https://www.chicagotribune.com/2026/11/18/cook...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
